# Gradient Boosting

**Objetivo:** construir o boosting **à mão** num problema 1D — árvores rasas ajustadas aos resíduos, um estágio por vez — e depois ver o efeito da taxa de aprendizado e do número de estágios com o `GradientBoostingClassifier`.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Boosting à mão: ajustar os resíduos

Começamos prevendo a média de `y`. A cada estágio, uma árvore rasa (um toco, `max_depth=1`) aprende o **resíduo** que sobrou, e somamos uma fração `nu` dela ao modelo. Laço explícito, sem caixa-preta.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

x = np.linspace(0, 1, 60)
y = np.sin(2 * np.pi * x) + np.random.normal(0, 0.15, size=x.shape)
X = x.reshape(-1, 1)

nu = 0.3
F = np.full_like(y, y.mean())      # previsao inicial: a media
arvores = []
erros = []
for estagio in range(40):
    residuo = y - F                # o que ainda falta
    toco = DecisionTreeRegressor(max_depth=1).fit(X, residuo)
    F = F + nu * toco.predict(X)    # incorpora uma fracao da nova arvore
    arvores.append(toco)
    erros.append(np.mean((y - F) ** 2))
print("erro (MSE) apos 1 estagio: ", round(erros[0], 3))
print("erro (MSE) apos 40 estagios:", round(erros[-1], 3))

## 2. O modelo tomando forma

Mostramos a previsão acumulada após 1, 5 e 40 estágios: de quase uma reta a uma boa aproximação da curva verdadeira.

In [ ]:
grade = np.linspace(0, 1, 200).reshape(-1, 1)
figura = go.Figure()
figura.add_trace(go.Scatter(x=x, y=y, mode="markers",
                            marker=dict(color=SUAVE, size=5), name="dados"))
figura.add_trace(go.Scatter(x=grade.ravel(), y=np.sin(2*np.pi*grade.ravel()),
                            mode="lines", line=dict(color=TINTA, dash="dash"), name="verdade"))
for n_estagios in [1, 5, 40]:
    pred = np.full(grade.shape[0], y.mean())
    for toco in arvores[:n_estagios]:
        pred = pred + nu * toco.predict(grade)
    figura.add_trace(go.Scatter(x=grade.ravel(), y=pred, mode="lines",
                                name=f"{n_estagios} estagios"))
figura.update_layout(title="Boosting a mao: o modelo se aproxima estagio a estagio",
                     height=400, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. Taxa de aprendizado e número de estágios

Agora com o `GradientBoostingClassifier` do scikit-learn, num problema de classificação. O `staged_predict` dá a previsão após cada estágio, então vemos o erro de treino e de teste em função do número de árvores, para duas taxas de aprendizado.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

dados = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(dados.data, dados.target,
                                          test_size=0.3, random_state=SEMENTE, stratify=dados.target)

figura = go.Figure()
for taxa, cor in [(0.1, AZUL), (1.0, VERMELHO)]:
    modelo = GradientBoostingClassifier(n_estimators=200, learning_rate=taxa,
                                        max_depth=2, random_state=SEMENTE)
    modelo.fit(X_tr, y_tr)
    erro_teste = []
    for previsto in modelo.staged_predict(X_te):
        erro_teste.append(1 - np.mean(previsto == y_te))
    figura.add_trace(go.Scatter(y=erro_teste, mode="lines",
                                line=dict(color=cor), name=f"taxa {taxa}"))
    print("taxa", taxa, "-> menor erro de teste:", round(min(erro_teste), 3),
          "no estagio", int(np.argmin(erro_teste)) + 1)
figura.update_layout(title="Erro de teste vs numero de estagios, por taxa de aprendizado",
                     xaxis_title="estagios", yaxis_title="erro de teste", height=360,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercício

Na figura do item 3, a taxa 1.0 atinge o menor erro mais cedo, mas costuma ficar mais irregular; a taxa 0.1 desce devagar e suave. Qual você usaria em produção e por quê?

<details><summary>Ver resposta</summary>

Em geral a **taxa menor (0,1)**, com mais estágios e *early stopping*. Passos pequenos regularizam (o *shrinkage*), dando uma curva de erro mais suave e estável e, tipicamente, melhor generalização. A taxa 1,0 chega rápido mas é sensível a ruído e pode passar do ponto. Troca-se um pouco de tempo de treino por robustez.

</details>